#  CASE 06 — Square 8×8 + 5×5 — Correlation Center

In [1]:
%%writefile case06_sq_5x5_corrCenter.cu

#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define WIDTH       8
#define HEIGHT      8
#define MASK_WIDTH  5
#define MASK_HEIGHT 5
#define BLOCK_SIZE  8

__global__ void correlationCenter(int *dA, int *dMask, int *dC,
                                  int width, int height,
                                  int mWidth, int mHeight)
{
    int col = threadIdx.x + blockIdx.x * blockDim.x;
    int row = threadIdx.y + blockIdx.y * blockDim.y;
    if (row < height && col < width)
    {
        int sum = 0;
        for (int i = 0; i < mHeight; i++)
            for (int j = 0; j < mWidth; j++)
            {
                int r = row + i - mHeight/2;   /* center */
                int c = col + j - mWidth/2;
                if (r>=0 && r<height && c>=0 && c<width)
                    sum += dA[r*width+c] * dMask[i*mWidth+j];
            }
        dC[row*width+col] = sum;
    }
}

void printMatrix(const char *label, int *M, int w, int h)
{
    printf("\n%s:\n", label);
    for (int r = 0; r < h; r++)
    {
        for (int c = 0; c < w; c++)
            printf("%6d", M[r*w+c]);
        printf("\n");
    }
}

int main()
{
    int size     = WIDTH * HEIGHT * sizeof(int);
    int maskSize = MASK_WIDTH * MASK_HEIGHT * sizeof(int);

    int *hA    = (int*) malloc(size);
    int *hMask = (int*) malloc(maskSize);
    int *hC    = (int*) malloc(size);

    srand(time(NULL));
    for (int i = 0; i < WIDTH*HEIGHT; i++)
        hA[i] = rand()%9+1;

    int tempMask[5][5] = {
        {1,1,1,1,1},
        {1,2,2,2,1},
        {1,2,4,2,1},
        {1,2,2,2,1},
        {1,1,1,1,1}
    };
    for (int i = 0; i < MASK_HEIGHT; i++)
        for (int j = 0; j < MASK_WIDTH; j++)
            hMask[i*MASK_WIDTH+j] = tempMask[i][j];

    printMatrix("Input Matrix (8x8)", hA,    WIDTH,      HEIGHT);
    printMatrix("Mask (5x5)",         hMask, MASK_WIDTH, MASK_HEIGHT);

    int *dA, *dMask, *dC;
    cudaMalloc((void**)&dA,    size);
    cudaMalloc((void**)&dMask, maskSize);
    cudaMalloc((void**)&dC,    size);
    cudaMemcpy(dA,    hA,    size,     cudaMemcpyHostToDevice);
    cudaMemcpy(dMask, hMask, maskSize, cudaMemcpyHostToDevice);

    dim3 DimBlock(BLOCK_SIZE, BLOCK_SIZE, 1);
    dim3 DimGrid((int)ceil((float)WIDTH/BLOCK_SIZE),
                 (int)ceil((float)HEIGHT/BLOCK_SIZE), 1);

    cudaEvent_t start, stop; float gpuTime;
    cudaEventCreate(&start); cudaEventCreate(&stop);
    cudaEventRecord(start);

    correlationCenter<<<DimGrid,DimBlock>>>(dA,dMask,dC,
                       WIDTH,HEIGHT,MASK_WIDTH,MASK_HEIGHT);

    cudaEventRecord(stop); cudaEventSynchronize(stop);
    cudaEventElapsedTime(&gpuTime, start, stop);
    cudaMemcpy(hC, dC, size, cudaMemcpyDeviceToHost);

    printMatrix("OUTPUT: Correlation Center (8x8, 5x5)", hC, WIDTH, HEIGHT);
    printf("\nGPU Time: %.4f ms\n", gpuTime);
    printf("Grid: %dx%d  Block: %dx%d\n",
            DimGrid.x,DimGrid.y,DimBlock.x,DimBlock.y);

    cudaFree(dA); cudaFree(dMask); cudaFree(dC);
    free(hA); free(hMask); free(hC);
    cudaEventDestroy(start); cudaEventDestroy(stop);
    return 0;
}

Writing case06_sq_5x5_corrCenter.cu


In [3]:
!nvcc -arch=sm_75 case06_sq_5x5_corrCenter.cu -o case06_sq_5x5_corrCenter

!./case06_sq_5x5_corrCenter


Input Matrix (8x8):
     7     8     5     6     3     1     9     6
     1     9     5     8     4     2     5     9
     1     7     8     2     3     8     2     3
     9     5     4     1     4     9     5     8
     7     9     2     9     9     2     6     8
     8     8     4     9     9     8     9     1
     6     5     9     6     3     1     8     9
     4     1     1     5     9     5     3     4

Mask (5x5):
     1     1     1     1     1
     1     2     2     2     1
     1     2     4     2     1
     1     2     2     2     1
     1     1     1     1     1

OUTPUT: Correlation Center (8x8, 5x5):
    90   118   128   122   101    97   105    86
   104   155   168   162   139   139   136   119
   121   176   201   176   169   190   161   121
   143   178   200   192   192   209   175   133
   152   193   202   213   218   207   183   140
   141   186   215   215   214   221   190   129
   108   149   189   181   178   180   158   116
    70    94   116   126   136   128